In [1]:
import cv2
import joblib
import numpy as np

from skimage.feature import hog

In [2]:
model = joblib.load("../model/hand-gesture-recognition-model.sav")

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [3]:
labels = {
    0: "VolumeDown",
    1: "Previous",
    2: "Next",
    3: "PlayPause",
    4: "VolumeUp"
}

In [4]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot Open Camera")

In [5]:
while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # -----------------------------
    # ROI Coordinates
    # -----------------------------

    x1 = 350
    y1 = 100

    x2 = 600
    y2 = 350

    roi = frame[y1:y2, x1:x2]

    # Draw Rectangle

    cv2.rectangle(frame,
                  (x1, y1),
                  (x2, y2),
                  (0,255,0),
                  2)

    # -----------------------------
    # Preprocessing
    # -----------------------------
    
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (128,128))
    
    # -----------------------------
    # Check whether ROI contains a hand
    # -----------------------------
    
    if gray.std() < 15:
    
        gesture = "Place Hand"
    
    else:
    
        # -----------------------------
        # HOG Feature
        # -----------------------------
    
        feature_vector = hog(gray)
        feature_vector = feature_vector.reshape(1,-1)
    
        # -----------------------------
        # Prediction
        # -----------------------------
    
        prediction = model.predict(feature_vector)
        gesture = labels[prediction[0]]
    
    # -----------------------------
    # Display Result
    # -----------------------------
    
    cv2.putText(frame,
                gesture,
                (20,50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,255,0),
                2)

    cv2.imshow("Hand Gesture Recognition", frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()